In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


How can the ethical frameworks of consequentialism and deontology be reconciled when designing autonomous AI systems that must make real-time decisions impacting human well-being?


In [8]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [9]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

A practical way to reconcile consequentialist and deontological ethics in real-time autonomous systems is to separate the decision problem into two complementary layers and enforce hard deontic constraints while optimizing for welfare within those constraints. This “hybrid” approach keeps non-negotiable duties (deontology) from being overridden by outcomes (consequentialism), yet still seeks to maximize well-being whenever it does not violate the duties.

Key ideas you can use

- Hard deontic constraints (what you must not do)
  - Encode duties, rights, and prohibitions as non-negotiable rules. Examples: do not harm pedestrians, respect autonomy and consent, do not disclose sensitive information, never take a life-saving action that violates fundamental rights, etc.
  - Represent these as hard constraints or forbidden actions in your decision module. If a proposed action would violate a deontic rule, discard it immediately.

- Consequentialist objective (what you try to maximize within the rules)
  - Define a welfare-related objective function (e.g., minimizing total expected harm, maximizing safety-adjusted well-being, fairness or equity metrics) that you optimize subject to the deontic constraints.
  - Treat the outcome objective as a soft constraint with measurable metrics (risk, harm, benefit to people, duration of impact, etc.) that you seek to improve as long as you stay within the hard constraints.

- Priority structure
  - Deontic constraints outrank outcomes. If there is a conflict, the system should not violate the hard rules and should escalate to human oversight, a safe default, or a fallback behavior.
  - In ambiguous cases, prefer conservative actions that preserve safety and rights, even if they slightly reduce predicted welfare.

- Shielding and runtime safety
  - Use a runtime “safety shield” that monitors actions proposed by a learning or planning module and vetoes any action that would violate deontic rules.
  - Implement fallback behaviors (e.g., safe stop, request human input) when no action can satisfy both sets of criteria.

- Formalization and reasoning
  - Represent obligations, permissions, and prohibitions with a deontic logic flavor (even if approximate) to keep rules explicit and verifiable.
  - Pair this with a utility or risk model for outcomes. Techniques include constrained optimization, multi-objective optimization, or risk-aware planning.

- Uncertainty and real-time constraints
  - Precompute or learn policies that respect hard constraints and are quick to evaluate at runtime.
  - Use fast approximate solvers or policy-shield hybrids so decisions can be made within latency requirements.
  - Incorporate uncertainty into both the deontic checks (e.g., “we might be uncertain about consent”) and the outcome estimates (probabilistic harm/risk).

A practical design blueprint

1) Clarify normative components
- List non-negotiable duties or rights relevant to the domain (e.g., do not physically harm people, protect privacy, obtain informed consent where applicable).
- Specify desirable welfare objectives (e.g., reduce harm, maximize safety for all affected parties, ensure equitable outcomes).

2) Formalize constraints and objectives
- Deontic layer: encode duties as hard constraints or forbidden actions.
- Consequential layer: define a welfare/risk objective to optimize (e.g., minimize expected harm, maximize life-years saved, balance risks among groups).

3) Architecture with a decision loop
- Perception and risk assessment: estimate who is affected, possible actions, and uncertainties.
- Deontic check: filter candidate actions to those that satisfy all hard constraints.
- Welfare optimization: among the deontically permissible actions, evaluate the welfare objective and select the best action.
- Safety shield and fallback: if no action satisfies the constraints or if confidence is too low, trigger safe defaults or human-in-the-loop escalation.

4) Learning and adaptation
- Train components with a focus on respecting constraints (penalize violations, use imitation data that reflects human-declared non-negotiables).
- Regularly audit decisions for deontic violations and adjust rules or thresholds as needed.

5) Verification, testing, and governance
- Scenario-based testing to stress both deontic compliance and welfare performance.
- Formal or semi-formal verification where feasible to prove that hard constraints are never violated under modeled conditions.
- Maintain audit logs and explainability: record which constraints were active, why a welfare-optimal choice was made, and how conflicts were resolved.
- Establish human-in-the-loop pathways for high-stakes or highly ambiguous cases.

Illustrative examples

- Autonomous vehicle
  - Deontic: never run over a pedestrian; never violate traffic laws; respect passenger autonomy (e.g., allow user overrides when appropriate).
  - Consequential: among safe options, minimize total expected harm to all parties, perhaps prioritizing the least-disadvantaged outcome.
  - Decision: if faced with a split-second choice where swerving protects a pedestrian but harms a passenger with high certainty, the hard constraint (do not hit a person) guides the action; if feasible, choose the option with lower overall expected harm within permissible maneuvers.

- Medical robot assistant
  - Deontic: respect patient autonomy and consent; do no harm; preserve privacy.
  - Consequential: maximize patient well-being, reduce risk of adverse events, ensure equitable care.
  - Decision: avoid actions that violate consent or autonomy, even if they would yield a marginally better outcome; otherwise optimize within those limits.

- Disaster response drone
  - Deontic: do not reveal sensitive locations; obtain authorization when required; protect civilians.
  - Consequential: deliver aid efficiently, minimize exposure to danger, maximize lives saved.
  - Decision: only take actions that comply with safety and privacy constraints; within those, plan routes that maximize aid delivery under real-time risk estimates.

Limitations and considerations

- Conflicts can be subtle and context-dependent; you may need escalation protocols or input from humans for edge cases.
- Value mis-specification: the chosen deontic rules and welfare metrics may not capture all ethical concerns; involve stakeholders and update rules as understanding evolves.
- Scalability: formalizing complex duties for all domains is hard; start with domain-specific, well-justified rule sets and iterate.
- Transparency and accountability: ensure decisions can be explained and traced to specific deontic constraints and welfare objectives.

In short, a reconciled design treats deontic constraints as non-negotiable guardrails and uses welfare-focused optimization inside those guardrails. This yields real-time, actionable behavior that respects fundamental duties while actively seeking to improve human well-being, with safeguards, transparency, and human oversight where necessary.

In [10]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Reconciling consequentialism and deontology in autonomous AI systems, especially when real-time decisions impact human well-being, is one of the most significant challenges in AI ethics. It's not about choosing one over the other, but rather about **integrating them in a layered, context-sensitive, and deliberative manner.**

Here's how this reconciliation can be approached:

### 1. Hierarchical or Layered Ethical Architectures

This is perhaps the most common and practical approach. Deontological rules act as **hard constraints or inviolable boundaries**, within which consequentialist optimization takes place.

*   **Deontological Outer Layer (Constraints):**
    *   **Definition:** These are universal moral duties or rights that the AI *must not* violate, regardless of the potential positive consequences. They represent the "red lines" or "non-negotiables."
    *   **Examples:**
        *   "Do not intentionally kill a human."
        *   "Do not intentionally cause grievous bodily harm."
        *   "Do not lie or deceive."
        *   "Do not discriminate based on protected characteristics (e.g., race, gender, religion)."
        *   "Always protect user privacy (unless legally compelled otherwise)."
    *   **Implementation:** These are hard-coded as **preconditions, absolute rules, or veto functions** in the AI's decision-making process. If a proposed action violates a deontological rule, it is immediately discarded, even if it leads to a seemingly better outcome in terms of utility.

*   **Consequentialist Inner Layer (Optimization):**
    *   **Definition:** Within the boundaries set by the deontological rules, the AI then evaluates potential actions based on their predicted outcomes, aiming to maximize overall utility, well-being, safety, efficiency, or other defined positive consequences.
    *   **Examples:**
        *   Minimizing harm (e.g., in an unavoidable accident, choose the path that causes the fewest severe injuries).
        *   Maximizing resource allocation (e.g., in a medical AI, optimize treatment plans for population health within resource constraints).
        *   Optimizing traffic flow to reduce accidents and congestion.
    *   **Implementation:** This involves **utility functions, reward systems (in reinforcement learning), risk assessments, and predictive models** that evaluate the likely impact of different actions.

**Example Scenario: Autonomous Vehicle**
*   **Deontological Constraint:** "Never intentionally steer into a pedestrian."
*   **Consequentialist Optimization:** If a collision is unavoidable (e.g., brake failure), the AI calculates the path that minimizes harm *among permissible options* (e.g., hitting a bollard vs. another car, or hitting a parked car vs. a car with occupants), *without* violating the "don't hit a pedestrian" rule.

### 2. Context-Dependent Prioritization

In certain extreme situations, the strict hierarchy might need to be re-evaluated, or the *dominant* framework might shift.

*   **High-Stakes, Unpredictable Scenarios:** When outcomes are highly uncertain, or the AI's predictive models might be unreliable, **deontological principles might gain higher priority** as a default safety mechanism. Sticking to "do no harm" when the consequences of action are unknown is often safer than trying to optimize an outcome based on faulty predictions.
*   **Emergency/Trolley-like Problems:** These are the most difficult. While ideally avoided by design, if an AI *must* choose between two "bad" outcomes, this is where the frameworks clash most dramatically. Here, a purely deontological stance might lead to inaction or a worse overall outcome, while a purely consequentialist stance might violate fundamental rights.
    *   **Reconciliation:** This often requires **pre-defined ethical protocols developed by human experts and regulators.** For example, "minimize loss of life," but with **constraints** like "do not sacrifice an innocent party *actively*." The "innocent party" constraint is deontological, while "minimize loss of life" is consequentialist. The *designers* have to make the difficult choice and embed it.

### 3. Consequentialism Informed by Deontology (and vice-versa)

The frameworks can influence each other's definitions of "good."

*   **Deontological Values as Utility Factors:** The "good" consequences that the AI optimizes for can include the **preservation of rights, fairness, and justice** as high-value factors in its utility function. So, a consequentialist calculation wouldn't just be about number of lives, but also *who* is affected, and whether fundamental rights are upheld in the process.
    *   *Example:* An AI might choose an outcome that saves 90 lives but ensures equitable risk distribution, over an outcome that saves 100 lives but disproportionately harms a vulnerable group (thus violating a fairness or non-discrimination duty).
*   **Consequentialist Rationale for Deontological Rules:** Many deontological rules (e.g., "don't lie," "keep promises") are upheld not just because they are inherently right, but also because their *long-term, societal consequences* are overwhelmingly positive (building trust, stability, cooperation). The AI could be designed with an understanding that these rules ultimately contribute to overall well-being.

### 4. Proactive Design & "Ethical Bounding"

Reconciliation isn't just a runtime decision; it's a fundamental part of the design process.

*   **Pre-computation of Ethical Scenarios:** Identify potential ethical dilemmas *before* deployment. Create decision trees or rulesets that reflect reconciled ethical judgments for known conflicts.
*   **Simulation and Testing:** Extensively test the AI in simulated environments that present ethical dilemmas. Analyze how it handles conflicts between deontological constraints and consequentialist goals.
*   **Human-in-the-Loop (HITL) for Ambiguity:** For decisions that fall into a grey area, or where the AI detects a significant conflict between its internal ethical frameworks, the system should flag the decision for human review or intervention. This ensures that complex ethical judgments remain ultimately subject to human deliberation.
*   **Transparency and Explainability:** The AI should be able to articulate *why* it made a particular decision, referencing both the deontological rules it adhered to and the consequentialist outcomes it sought to achieve. This builds trust and allows for auditability.

### 5. Training Data and Learning

Ethical principles can be embedded in the AI's learning process.

*   **Reinforcement Learning with Ethical Rewards/Penalties:** Design reward functions that strongly penalize violations of deontological rules and reward actions that align with positive consequences. For instance, a very large negative reward for violating a "do no harm" rule.
*   **Curated Datasets:** Train the AI on datasets that exemplify ethical decision-making in complex scenarios, showing how humans have (or should have) reconciled these frameworks.

### Challenges and Considerations

*   **Quantifying Deontological Values:** How do you numerically represent "dignity" or "justice" in an AI's utility function?
*   **Conflicting Duties:** What if two deontological rules conflict (e.g., duty to protect life vs. duty to tell the truth, if the truth puts life at risk)? This requires human-defined priority rules.
*   **The "Good" is Subjective:** Different societies, cultures, and individuals have different ethical weightings. Which set of ethics should the AI adhere to? This points to the need for public deliberation and regulation.
*   **Unforeseen Circumstances:** No matter how well designed, AI will encounter novel situations where its pre-programmed reconciliation strategies might fail or lead to unexpected outcomes.

### Conclusion

Reconciling consequentialism and deontology in autonomous AI is not about finding a single, universal algorithm, but rather about developing a **robust, multi-layered ethical architecture.** It primarily involves setting **deontological "guardrails"** that define non-negotiable boundaries, and then allowing **consequentialist optimization** within those bounds. Crucially, this reconciliation must be an ongoing process, involving proactive design, rigorous testing, transparency, and a persistent **human-in-the-loop** for the most complex and ambiguous ethical dilemmas. It's a continuous negotiation between what *must not be done* and what *should achieve the best outcome*.

In [11]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Consequentialism and deontology are two prominent ethical frameworks in philosophy, which may seem conflicting at first glance. However, they can be reconciled when designing autonomous AI systems by considering the following approaches:

1. **Hybrid Ethics**: Combine elements of consequentialism and deontology to create a hybrid approach. This framework considers both the consequences of an action and its adherence to moral rules (duties). For example, an AI system might evaluate the potential harm or benefit of an action and then decide whether it aligns with established moral guidelines.
2. **Prioritizing Shared Human Values**: Align the AI's goals and decision-making processes with human values that are widely accepted as moral principles, such as respect for human life, dignity, and well-being. This can help ensure that the AI system makes decisions in a way that is consistent with widely-held ethical norms.
3. **Framework of Moral Probabilities**: Develop an integrated framework that considers both the likelihood of outcomes (consequentialist) and the moral probability of certain actions (deontological). For example, an AI system might weigh the potential benefits against potential risks and consider whether the action aligns with established moral standards.
4. **Prioritizing Human Well-being in Consequentialism**: Adapt consequentialism to prioritize human well-being over other outcomes. By valuing human flourishing and autonomy, the AI system can make decisions that benefit human society while minimizing harm.
5. **Utilitarianism with constraints**: Implement a utilitarian approach by maximizing overall happiness or well-being but, alongside the consequences consideration, implement deontological constraints, which maintain the moral principles in decision-making processes.

In practice:

- Develop clearly defined objectives and constraints for AI systems
- Regularly update and refine these to ensure alignment with evolving human values and societal norms
- Implement hybrid decision-making models that incorporate multiple types of evaluation and feedback

Balancing consequentialism and deontology requires careful consideration, research, testing, validation and continuous improvement. By exploring the intersections between these ethical frameworks and incorporating elements from both in decision-making processes, we can design autonomous AI systems that prioritize human well-being while avoiding morally questionable outcomes.

**Challenges Ahead:**

1. Overcoming value conflicts: Different individuals may have varying moral values; and different perspectives on how to balance consequentialism and deontology.
2. Complexity, uncertainty, and risk in high-stakes AI decision-making contexts (e.g., life-and-death situations)
3. Rapid changes in human society and evolving normative framework

**Future directions for research:**

1. In-depth exploration of frameworks and methodologies integrating both consequentialist and deontological principles
2. Evaluation of the performance, robustness and reliability of these approaches
3. Integration into other AI components (e.g., decision assistance systems for clinicians, medical robots)
4. Addressing challenges specific to AI decision-making systems

In [12]:
# So where are we?

print(competitors)
print(answers)


['gpt-5-nano', 'gemini-2.5-flash', 'llama3.2']
['A practical way to reconcile consequentialist and deontological ethics in real-time autonomous systems is to separate the decision problem into two complementary layers and enforce hard deontic constraints while optimizing for welfare within those constraints. This “hybrid” approach keeps non-negotiable duties (deontology) from being overridden by outcomes (consequentialism), yet still seeks to maximize well-being whenever it does not violate the duties.\n\nKey ideas you can use\n\n- Hard deontic constraints (what you must not do)\n  - Encode duties, rights, and prohibitions as non-negotiable rules. Examples: do not harm pedestrians, respect autonomy and consent, do not disclose sensitive information, never take a life-saving action that violates fundamental rights, etc.\n  - Represent these as hard constraints or forbidden actions in your decision module. If a proposed action would violate a deontic rule, discard it immediately.\n\n- Co

In [13]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-5-nano

A practical way to reconcile consequentialist and deontological ethics in real-time autonomous systems is to separate the decision problem into two complementary layers and enforce hard deontic constraints while optimizing for welfare within those constraints. This “hybrid” approach keeps non-negotiable duties (deontology) from being overridden by outcomes (consequentialism), yet still seeks to maximize well-being whenever it does not violate the duties.

Key ideas you can use

- Hard deontic constraints (what you must not do)
  - Encode duties, rights, and prohibitions as non-negotiable rules. Examples: do not harm pedestrians, respect autonomy and consent, do not disclose sensitive information, never take a life-saving action that violates fundamental rights, etc.
  - Represent these as hard constraints or forbidden actions in your decision module. If a proposed action would violate a deontic rule, discard it immediately.

- Consequentialist objective (what yo

In [14]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [15]:
print(together)

# Response from competitor 1

A practical way to reconcile consequentialist and deontological ethics in real-time autonomous systems is to separate the decision problem into two complementary layers and enforce hard deontic constraints while optimizing for welfare within those constraints. This “hybrid” approach keeps non-negotiable duties (deontology) from being overridden by outcomes (consequentialism), yet still seeks to maximize well-being whenever it does not violate the duties.

Key ideas you can use

- Hard deontic constraints (what you must not do)
  - Encode duties, rights, and prohibitions as non-negotiable rules. Examples: do not harm pedestrians, respect autonomy and consent, do not disclose sensitive information, never take a life-saving action that violates fundamental rights, etc.
  - Represent these as hard constraints or forbidden actions in your decision module. If a proposed action would violate a deontic rule, discard it immediately.

- Consequentialist objective (w

In [16]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [17]:
print(judge)

You are judging a competition between 3 competitors.
Each model has been given this question:

How can the ethical frameworks of consequentialism and deontology be reconciled when designing autonomous AI systems that must make real-time decisions impacting human well-being?

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

A practical way to reconcile consequentialist and deontological ethics in real-time autonomous systems is to separate the decision problem into two complementary layers and enforce hard deontic constraints while optimizing for welfare within those constraints. This “hybrid” approach keeps non-negotiable duties (deontology) from being overridden by outcomes 

In [18]:
judge_messages = [{"role": "user", "content": judge}]

In [19]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["1", "2", "3"]}


In [20]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gpt-5-nano
Rank 2: gemini-2.5-flash
Rank 3: llama3.2
